# Computation of Monte-Carlo load cases under the assumption of Binomial distributions per room


Load the general room type dependent demands and the specific building demands

In [ ]:
# set activate directory to repository's root
import os
os.chdir("../../../")
!pwd

In [ ]:
import pandas as pd

from vensys_clustering import load_yaml, compute_required_volume_flows, build_zone_pmfs, save_scenario_data_to_yaml

from vensys_clustering.load_case_sampling import count_unique_rows, sample_zone_pmfs_sobol, sampled_load_cases_to_dict
from importlib.resources import files

standard_data = load_yaml(files("vensys_clustering.data").joinpath("general.yml"))

PATH_TO_RAW_SCENARIO_FILE = "data/load_case_data/raw_GPZ_load_cases.yml"

NUMBER_OF_IDENTICAL_FLOORS = 1 # 16 for office tower, 1 for multipurpose building (GPZ)

raw_lc_data = load_yaml(PATH_TO_RAW_SCENARIO_FILE)
rooms_to_merge = raw_lc_data["rooms_to_merge"]

In [ ]:
def expand_identical_floors(
    df: pd.DataFrame,
    rooms_to_merge: dict[str, list[str]],
    n_floors: int,
) -> tuple[pd.DataFrame, dict[str, list[str]]]:
    """
    Duplicate all rooms and room-merge mappings for identical floors.
    """
    if n_floors <= 1:
        return df, rooms_to_merge

    df_expanded = []
    rooms_to_merge_expanded = {}

    for floor_idx in range(n_floors):
        prefix = str(floor_idx)

        df_copy = df.copy()
        df_copy["room"] = prefix + df_copy["room"].astype(str)

        rooms_to_merge_expanded.update(
            {
                prefix + zone: [prefix + room for room in rooms]
                for zone, rooms in rooms_to_merge.items()
            }
        )

        df_expanded.append(df_copy)

    return pd.concat(df_expanded, ignore_index=True), rooms_to_merge_expanded

compute volume flows

In [ ]:
df = compute_required_volume_flows(standard_data, raw_lc_data, overview_flag=True, include_revision=False)

merge the zone's binomial distributions

In [ ]:
zone_pmfs_by_time = build_zone_pmfs(df, rooms_to_merge)

In [ ]:
df, rooms_to_merge = expand_identical_floors(df, rooms_to_merge, NUMBER_OF_IDENTICAL_FLOORS)

sample from the pmfs using sobol sequence with `num_samples` being the amount of samples used per hour of the day

In [ ]:
room_samples, hours_vec = sample_zone_pmfs_sobol(zone_pmfs_by_time, num_samples=128)
room_samples = {room: arr.ravel() for room, arr in room_samples.items()}

count duplicates and merge them, including adding up their frequency

In [ ]:
row_counts_df = count_unique_rows(pd.DataFrame(room_samples))
row_counts_df = row_counts_df.loc[row_counts_df.drop(columns="frequency").sum(axis=1).sort_values().index].reset_index(drop=True) # sort ascending with sum of volume flow

bring in the correct format and save

In [ ]:
result_dict_sobol = sampled_load_cases_to_dict(row_counts_df)

In [ ]:
save_scenario_data_to_yaml(result_dict_sobol, "data/load_case_data/GPZ_monte_carlo_load_cases_128.yml")